In [1]:
import warnings

warnings.filterwarnings('ignore')

In [2]:
from utils.seed import SeedSetter
from utils.datasets import TimeSeriesDataset
from torch.utils.data import DataLoader

In [3]:
from gdmh.mia import PolynomType


class Config:
    seed: int = 42
    test_size: float = 0.2
    k_best: int = 3
    polynomial_type: PolynomType = PolynomType.QUADRATIC
    limit: float = 0.01
    max_iters: int = 100

In [4]:
SeedSetter(Config.seed)

Random seed set to 42


In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [6]:
data = pd.read_csv("../data/turbines_train.csv")
data = data.drop(["Unnamed: 0", "Time"], axis=1)

In [7]:
data.head(5)

,Location,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Power
0,1,28.2796,84.664205,24.072595,1.605389,1.267799,145.051683,161.057315,1.336515,0.163496
1,1,28.1796,85.664205,24.272595,2.225389,3.997799,150.051683,157.057315,4.336515,0.142396
2,1,26.5796,90.664205,24.072595,1.465389,2.787799,147.051683,149.057315,3.136515,0.121396
3,1,27.1796,87.664205,23.872595,1.465389,2.697799,57.051683,104.057315,1.536515,0.100296
4,1,27.0796,87.664205,23.672595,2.635389,4.437799,57.051683,83.057315,3.936515,0.079296


In [8]:
data["Location"].unique()

array([1, 2, 3, 4])

In [9]:
scaler = StandardScaler()
features = data.drop(['Power', 'Location'], axis=1)
scaled_features = scaler.fit_transform(features)
scaled_data = pd.DataFrame(scaled_features, columns=features.columns)
scaled_data['Power'] = data['Power'].values
scaled_data['Location'] = data['Location'].values

In [10]:
scaled_data['Location'] = pd.Categorical(scaled_data['Location']).codes
scaled_data.head()

,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Power,Location
0,-0.804021,0.837857,-0.562194,-1.355386,-1.850563,-0.555885,-0.391588,-1.845008,0.163496,0
1,-0.808581,0.896680,-0.552661,-1.049536,-0.957421,-0.505925,-0.431150,-1.017793,0.142396,0
2,-0.881539,1.190795,-0.562194,-1.424449,-1.353283,-0.535901,-0.510276,-1.348679,0.121396,0
3,-0.854179,1.014326,-0.571727,-1.424449,-1.382727,-1.435186,-0.955355,-1.789861,0.100296,0
4,-0.858739,1.014326,-0.581260,-0.847281,-0.813472,-1.435186,-1.163059,-1.128089,0.079296,0


In [11]:
scaled_data['power_log'] = np.log1p(scaled_data['Power'])
scaled_data = scaled_data.drop(['Power'], axis=1)
scaled_data.head()

,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Location,power_log
0,-0.804021,0.837857,-0.562194,-1.355386,-1.850563,-0.555885,-0.391588,-1.845008,0,0.151429
1,-0.808581,0.896680,-0.552661,-1.049536,-0.957421,-0.505925,-0.431150,-1.017793,0,0.133128
2,-0.881539,1.190795,-0.562194,-1.424449,-1.353283,-0.535901,-0.510276,-1.348679,0,0.114574
3,-0.854179,1.014326,-0.571727,-1.424449,-1.382727,-1.435186,-0.955355,-1.789861,0,0.095579
4,-0.858739,1.014326,-0.581260,-0.847281,-0.813472,-1.435186,-1.163059,-1.128089,0,0.076309


In [12]:
print(
    f"features shape: {scaled_data.iloc[:, :-1].to_numpy().shape}, target shape: {scaled_data.iloc[:, -1].to_numpy().shape}")

features shape: (140160, 9), target shape: (140160,)


In [13]:
from sklearn.model_selection import train_test_split

y = scaled_data['power_log']
X = scaled_data.drop(columns='power_log')

train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=Config.seed)

In [14]:
def criterion(y_true, y_pred) -> float:
    return np.mean((y_true - y_pred) ** 2)

In [15]:
from sklearn.metrics import mean_absolute_error, r2_score

metrics = {
    "r2_score": r2_score,
    "mae": mean_absolute_error,
}

In [16]:
from itertools import product

test_size_values = [0.6, 0.7, 0.8, 0.9]
k_best_values = [4, 5, 6, 7]
polynomial_types = [PolynomType.LINEAR, PolynomType.PARTIAL_QUADRATIC, PolynomType.QUADRATIC]
param_grid = list(product(test_size_values, k_best_values, polynomial_types))

In [17]:
from gdmh.fuzzy_gmdh import FuzzyGMDH
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


for test_size, k_best, polynomial_type in param_grid:
    print(f"Testing FuzzyGMDH with hyperparams: test_size={test_size}, k_best={k_best}, polynomial_type={polynomial_type}")
    fuzzy_gmdh = FuzzyGMDH(test_size=test_size, 
              k_best=k_best,
              polynomial_type=polynomial_type,
              criterion=criterion,
              seed=Config.seed,
              max_iters=Config.max_iters,
              limit=Config.limit,
              metrics=metrics
    )
    fuzzy_gmdh.fit(train_x.to_numpy(), train_y.to_numpy())
    test_predictions = fuzzy_gmdh.predict(test_x.to_numpy())
    mse = mean_squared_error(test_y.to_numpy(), test_predictions)
    mae = mean_absolute_error(test_y.to_numpy(), test_predictions)
    r2 = r2_score(test_y.to_numpy(), test_predictions)
    
    print(f"Test Mean Squared Error (MSE): {mse:.4f}")
    print(f"Test Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Test R2 Score: {r2:.4f}")

    print(f"="*50)

Testing FuzzyGMDH with hyperparams: test_size=0.6, k_best=4, polynomial_type=PolynomType.LINEAR
Iteration 1 completed: {'iter': 1, 'avg_test_quality': 0.0424, 'avg_y_pred_test': 0.343, 'avg_test_r2_score': -0.2304, 'avg_test_mae': 0.1801}
Iteration 2 completed: {'iter': 2, 'avg_test_quality': 0.0424, 'avg_y_pred_test': 0.343, 'avg_test_r2_score': -0.2305, 'avg_test_mae': 0.1801}
Stopping: improvement below limit.
Test Mean Squared Error (MSE): 1.0718
Test Mean Absolute Error (MAE): 0.8434
Test R2 Score: -30.2400
Testing FuzzyGMDH with hyperparams: test_size=0.6, k_best=4, polynomial_type=PolynomType.PARTIAL_QUADRATIC
Iteration 1 completed: {'iter': 1, 'avg_test_quality': 0.0422, 'avg_y_pred_test': 0.3422, 'avg_test_r2_score': -0.225, 'avg_test_mae': 0.1797}
Iteration 2 completed: {'iter': 2, 'avg_test_quality': 0.0421, 'avg_y_pred_test': 0.3418, 'avg_test_r2_score': -0.2215, 'avg_test_mae': 0.1795}
Stopping: improvement below limit.
Test Mean Squared Error (MSE): 2879745.0805
Test Mean